# Phase 10 — Causal Five-Fold Cross-Validation

**Objective:** Train the frozen full causal method from the Phase 8 pilot on
all five fixed BUSI folds. Compare against the Phase 5 baseline using paired
out-of-fold predictions.

**Rules:**
- Use `reports/results/frozen_causal_configuration.yaml` without modifying
  its scientific settings.
- Use the same split, backbone, initialization policy, transforms, training
  budget, and evaluation code as the baseline except for declared causal terms.
- For each fold: train on training only, select checkpoint on validation only,
  select threshold on validation only, evaluate test once, preserve all run
  artifacts.
- Keep background donors partition-local.
- Support Colab resume. Never overwrite a completed run.
- Save per-sample predictions with causal components.
- Validate each fold before aggregation.
- After all five folds aggregate out-of-fold predictions and compare against
  the baseline using paired samples.
- Do not load BUS-UCLM.

**Phase 10 gate:**
- All five causal folds have validated terminal states.
- The configuration remained frozen.
- Baseline and causal denominators match.
- Donor-leakage tests pass.
- BUS-UCLM was not loaded.

## 10.0 — Colab bootstrap

Detects Google Colab and clones/pulls the repository. In VS Code, does nothing.

In [2]:
import os
from pathlib import Path


def is_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


REPO_URL = "https://github.com/Sayem7456/CausalMask-XAI.git"
COLAB_TARGET = Path("/content/CausalMask-XAI")

if is_colab():
    print("Detected Google Colab environment.")
    if COLAB_TARGET.exists() and (COLAB_TARGET / "CausalMask-XAI.md").exists():
        print(f"Repository present at {COLAB_TARGET}. Pulling latest...")
        !cd {COLAB_TARGET} && git pull --ff-only
        print("Repository updated to latest commit.")
    else:
        if COLAB_TARGET.exists():
            import shutil
            shutil.rmtree(COLAB_TARGET)
        print(f"Cloning repository from {REPO_URL}...")
        !git clone {REPO_URL} {COLAB_TARGET}
        assert (COLAB_TARGET / "CausalMask-XAI.md").exists(), "Clone failed: marker file missing"
    os.environ["CAUSALMASK_PROJECT_ROOT"] = str(COLAB_TARGET)
    !cd {COLAB_TARGET} && pip install -e .[dev] --quiet 2>&1 | tail -3
    print("Package installed in editable mode.")
else:
    print("Not in Colab — skipping bootstrap.")

Detected Google Colab environment.
Repository present at /content/CausalMask-XAI. Pulling latest...
Already up to date.
Repository updated to latest commit.
Package installed in editable mode.


## 10.1 — Resolve project root

Resolution order:
1. `CAUSALMASK_PROJECT_ROOT` environment variable
2. Walk up from cwd looking for `CausalMask-XAI.md`
3. Colab fallback `/content/CausalMask-XAI`

In [3]:
import os
import sys
from pathlib import Path


def _resolve_project_root() -> Path:
    env_root = os.environ.get("CAUSALMASK_PROJECT_ROOT")
    if env_root:
        p = Path(env_root)
        if (p / "CausalMask-XAI.md").exists():
            return p.resolve()
    cwd = Path.cwd()
    for candidate in [cwd] + list(cwd.parents):
        if (candidate / "CausalMask-XAI.md").exists():
            return candidate.resolve()
    colab_fallback = Path("/content/CausalMask-XAI")
    if colab_fallback.exists() and (colab_fallback / "CausalMask-XAI.md").exists():
        return colab_fallback.resolve()
    raise RuntimeError(
        "Cannot resolve project root. Set CAUSALMASK_PROJECT_ROOT or run from within the repo."
    )


PROJECT_ROOT = _resolve_project_root()
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
assert (PROJECT_ROOT / "CausalMask-XAI.md").exists(), "Marker file missing"

src_dir = str(PROJECT_ROOT / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
print(f"src dir added to path: {src_dir}")

PROJECT_ROOT = /content/CausalMask-XAI
src dir added to path: /content/CausalMask-XAI/src


## 10.2 — Load frozen configuration and display

Load the frozen causal configuration from the Phase 8 pilot. This
configuration MUST NOT be modified. Every value is burned into the
five-fold causal run directories.

BUS-UCLM must not influence any choice.

In [4]:
import json
from datetime import datetime, timezone

import torch
import yaml

from causalmask.reproducibility import capture_environment, configure_reproducibility

SEED = 42
repro_info = configure_reproducibility(seed=SEED)
env_info = capture_environment(project_root=PROJECT_ROOT)

FROZEN_CONFIG_PATH = PROJECT_ROOT / "reports" / "results" / "frozen_causal_configuration.yaml"
assert FROZEN_CONFIG_PATH.exists(), f"Frozen config not found: {FROZEN_CONFIG_PATH}"

with open(FROZEN_CONFIG_PATH) as f:
    frozen_raw = yaml.safe_load(f)

FROZEN_CFG = frozen_raw["config"]

print("Frozen configuration loaded from Phase 8 pilot.")
print(f"  Config frozen on: {frozen_raw['metadata']['freeze_date_utc']}")
print(f"  Pilot run: {frozen_raw['metadata']['pilot_run_id']}")

N_FOLDS = 5
FROZEN_CFG["n_folds"] = N_FOLDS

print(json.dumps(FROZEN_CFG, indent=2, default=str))

Frozen configuration loaded from Phase 8 pilot.
  Config frozen on: 2026-07-29T00:00:00.000000+00:00
  Pilot run: causal_full_effb0_fold0_seed42_pilot
{
  "amp_enabled": true,
  "augmentation": {
    "affine_scale_max": 1.05,
    "affine_scale_min": 0.95,
    "affine_translate_max": 0.05,
    "contrast_range": [
      0.9,
      1.1
    ],
    "gamma_range": [
      0.9,
      1.1
    ],
    "horizontal_flip_prob": 0.5,
    "noise_std": 0.005,
    "rotation_degrees": 10.0
  },
  "backbone": "efficientnet_b0",
  "batch_size": 16,
  "binary_classes": [
    "benign",
    "malignant"
  ],
  "causal_loss": {
    "background_weight": 0.5,
    "ce_weight": 1.0,
    "loss_variant": "full",
    "necessity_confidence_threshold": 0.6,
    "necessity_margin": 0.1,
    "necessity_ramp_epochs": 3,
    "necessity_warmup_epochs": 3,
    "necessity_weight": 0.5,
    "sufficiency_weight": 0.5,
    "use_detached_teacher": true
  },
  "classification_threshold_policy": "Youden's J statistic computed from 

## 10.3 — Mount Drive & restore Phase 2/3/5 artifacts

Mount Google Drive for persistent storage. Restores manifests, splits,
extracted data, and baseline checkpoints from Drive if missing locally.

In [5]:
import shutil
import zipfile

MANIFESTS_DIR = PROJECT_ROOT / "data" / "manifests"
SPLITS_DIR = PROJECT_ROOT / "data" / "splits"
REPORTS_DIR = PROJECT_ROOT / "reports"
PHASES_DIR = PROJECT_ROOT / "artifacts" / "phases"
RUNS_DIR = PROJECT_ROOT / "artifacts" / "runs"
ARCHIVES_DIR = PROJECT_ROOT / "data" / "raw" / "archives"
EXTRACT_DIR = PROJECT_ROOT / "data" / "raw" / "extracted"
RESULTS_DIR = REPORTS_DIR / "results"

for d in [MANIFESTS_DIR, SPLITS_DIR, REPORTS_DIR, PHASES_DIR, RUNS_DIR,
          ARCHIVES_DIR, EXTRACT_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Manifests dir:  {MANIFESTS_DIR}")
print(f"Splits dir:     {SPLITS_DIR}")
print(f"Runs dir:       {RUNS_DIR}")

DRIVE_BASE = None
if is_colab():
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_BASE = Path("/content/drive/MyDrive/CausalMask-XAI")
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    print(f"Drive mounted. Artifacts will sync to {DRIVE_BASE}")
else:
    print("Not in Colab — Drive not mounted. Artifacts saved locally only.")


def restore_from_drive(subdir, filename, local_dir):
    if DRIVE_BASE is None:
        return False
    src = DRIVE_BASE / subdir / filename
    dst = local_dir / filename
    if dst.exists():
        return False
    if not src.exists():
        print(f"  [WARN] Not on Drive: {src}")
        return False
    local_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print(f"  Restored: {dst}")
    return True


def save_to_drive(src, subdir):
    if DRIVE_BASE is None:
        return False
    dst = DRIVE_BASE / subdir / src.name
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    return True


def save_dir_to_drive(src_dir, subdir):
    if DRIVE_BASE is None:
        return 0
    dst_base = DRIVE_BASE / subdir / src_dir.name
    count = 0
    for f in src_dir.rglob("*"):
        if f.is_file():
            rel = f.relative_to(src_dir)
            dst = dst_base / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dst)
            count += 1
    if count > 0:
        print(f"  Synced {count} files to Drive: {dst_base}")
    return count


print("\n--- Restoring Phase 2/3 artifacts from Drive ---")
for fname in [
    f"busi_manifest_{FROZEN_CFG['manifest_version']}.parquet",
    "busi_manifest_v2_grouped.parquet",
]:
    restore_from_drive("manifests", fname, MANIFESTS_DIR)

restore_from_drive("splits", f"{FROZEN_CFG['split_name']}.json", SPLITS_DIR)

for ds_name, cfg in FROZEN_CFG.get("datasets", {}).items():
    archive_path = ARCHIVES_DIR / Path(cfg["archive_rel"]).name
    restore_from_drive("archives", archive_path.name, ARCHIVES_DIR)
    extract_path = PROJECT_ROOT / cfg["extract_rel"]
    if not extract_path.exists() or not any(extract_path.iterdir()):
        if archive_path.exists():
            print(f"  {ds_name}: extracting from archive...")
            extract_path.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(archive_path, "r") as zf:
                zf.extractall(extract_path)
            print(f"  {ds_name}: extracted to {extract_path}")
        else:
            print(f"  {ds_name}: no archive found at {archive_path}.")
    else:
        print(f"  {ds_name}: extracted data already present at {extract_path}")

print("--- Restoring Phase 5 baseline predictions from Drive ---")
for fname in ["baseline_internal_predictions.parquet", "baseline_internal_metrics.json"]:
    restore_from_drive("reports/results", fname, RESULTS_DIR)
print("--- Restore complete ---\n")

Mounted at /content/drive
Drive mounted. Artifacts will sync to /content/drive/MyDrive/CausalMask-XAI

--- Restoring Phase 2/3 artifacts from Drive ---
  Restored: /content/CausalMask-XAI/data/manifests/busi_manifest_v2_grouped.parquet
  Restored: /content/CausalMask-XAI/data/splits/busi_binary_grouped_5fold_v1.json
  Restored: /content/CausalMask-XAI/data/raw/archives/breast-ultrasound-images-dataset.zip
  busi: extracting from archive...
  busi: extracted to /content/CausalMask-XAI/data/raw/extracted/busi
  Restored: /content/CausalMask-XAI/data/raw/archives/bus-uclm-breast-ultrasound-dataset.zip
  bus_uclm: extracting from archive...
  bus_uclm: extracted to /content/CausalMask-XAI/data/raw/extracted/bus_uclm
--- Restoring Phase 5 baseline predictions from Drive ---
  Restored: /content/CausalMask-XAI/reports/results/baseline_internal_predictions.parquet
--- Restore complete ---



## 10.4 — Verify split and manifest integrity

Before training, confirm the split digest matches. Check that real
extracted BUSI images are available. If either is missing, the
notebook cannot run a real experiment and is **blocked**.

In [6]:
import hashlib

import pandas as pd

from causalmask.data.splits import load_split, compute_split_digest, compute_manifest_digest

SPLIT_PATH = SPLITS_DIR / f"{FROZEN_CFG['split_name']}.json"

V2_PATH = MANIFESTS_DIR / "busi_manifest_v2_grouped.parquet"
V1_PATH = MANIFESTS_DIR / f"busi_manifest_{FROZEN_CFG['manifest_version']}.parquet"

if V2_PATH.exists():
    MANIFEST_PATH = V2_PATH
    manifest_version = "v2_grouped"
elif V1_PATH.exists():
    MANIFEST_PATH = V1_PATH
    manifest_version = "v1"
else:
    MANIFEST_PATH = None
    manifest_version = None

USE_REAL_DATA = SPLIT_PATH.exists() and MANIFEST_PATH is not None

if USE_REAL_DATA:
    split = load_split(SPLIT_PATH)
    split_digest = compute_split_digest(split)
    stored_digest = split.get("metadata", {}).get("split_digest", "")
    digest_match = split_digest == stored_digest
    print(f"Split loaded: {SPLIT_PATH.name}")
    print(f"  Stored digest:   {stored_digest[:16]}...")
    print(f"  Computed digest: {split_digest[:16]}...")
    print(f"  Digest match: {digest_match}")
    if not digest_match:
        raise RuntimeError("SPLIT DIGEST MISMATCH — do not proceed with this split.")
    print("Split integrity: PASSED")

    manifest_df = pd.read_parquet(MANIFEST_PATH)
    manifest_digest = compute_manifest_digest(manifest_df)
    print(f"Manifest loaded: {len(manifest_df)} samples (version: {manifest_version})")
    print(f"  Manifest digest: {manifest_digest[:16]}...")

    BUSI_EXTRACT = PROJECT_ROOT / FROZEN_CFG["datasets"]["busi"]["extract_rel"]
    HAS_REAL_IMAGES = BUSI_EXTRACT.exists() and any(BUSI_EXTRACT.iterdir())
    print(f"  Real BUSI images extracted: {HAS_REAL_IMAGES}")
    if not HAS_REAL_IMAGES:
        USE_REAL_DATA = False
else:
    print("Split or manifest not found. Real data NOT available. BLOCKED.")
    split = None
    split_digest = "blocked_no_real_data"
    manifest_digest = "blocked_no_real_data"
    HAS_REAL_IMAGES = False
    manifest_df = None

# Compute SHA-256 digest of the frozen config for provenance
with open(FROZEN_CONFIG_PATH, "rb") as f:
    frozen_config_bytes = f.read()
frozen_config_digest = hashlib.sha256(frozen_config_bytes).hexdigest()
print(f"\nFrozen config digest: {frozen_config_digest[:16]}...")
print(f"USE_REAL_DATA = {USE_REAL_DATA}")

Split loaded: busi_binary_grouped_5fold_v1.json
  Stored digest:   2a88e7ada1aff73e...
  Computed digest: 2a88e7ada1aff73e...
  Digest match: True
Split integrity: PASSED
Manifest loaded: 780 samples (version: v2_grouped)
  Manifest digest: 6462d283b3fcfe66...
  Real BUSI images extracted: True

Frozen config digest: 3dcc174d9f9fdd5b...
USE_REAL_DATA = True


## 10.5 — Define counterfactual generation functions

Two functions:
1. **Training** counterfactuals — batched, hard-coded, BACKGROUND SWAP DISABLED
   (same deviation as the pilot).
2. **Evaluation** counterfactuals — per-sample, includes background swap with
   partition-local donors for computing causal component metrics.

CRITICAL: Dataloader images are ImageNet-normalized. Before OpenCV processing
we must unnormalize back to [0, 1] range, convert to uint8, then re-normalize
after counterfactual generation.

Training donors come ONLY from the training partition.
Evaluation donors come ONLY from the test partition.

In [7]:
import numpy as np
import cv2
import torch

from causalmask.counterfactuals.masks import lesion_plus_margin, MarginConfig
from causalmask.counterfactuals.sufficient import generate_lesion_sufficient, SufficientConfig
from causalmask.counterfactuals.removal import generate_lesion_removed, RemovalConfig, RemovalOperator
from causalmask.counterfactuals.background_swap import generate_background_swap, SwapConfig

CF_CONFIG = FROZEN_CFG["counterfactual"]

margin_cfg = MarginConfig(margin_ratio=CF_CONFIG["margin_ratio"])
suff_cfg = SufficientConfig(
    margin_config=margin_cfg,
    blur_sigma=CF_CONFIG["blur_sigma"],
    use_feathered_blend=CF_CONFIG["feathered_blend"],
)
removal_op = RemovalOperator(CF_CONFIG["removal_operator"])
removal_cfg = RemovalConfig(margin_config=margin_cfg, operator=removal_op)
swap_cfg = SwapConfig(
    margin_config=margin_cfg,
    donor_class=CF_CONFIG["donor_class"],
    use_feathered_blend=CF_CONFIG["feathered_blend"],
    align_histogram=CF_CONFIG["align_histogram"],
    seed=SEED,
)

# ImageNet normalization stats (reversed for OpenCV counterfactual processing)
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float64)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float64)
IMAGENET_MEAN_T = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD_T = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def _unnormalize_tensor_to_numpy(img_tensor):
    """Unnormalize a [C,H,W] ImageNet-normalized tensor to uint8 numpy for OpenCV."""
    x = img_tensor.cpu()
    x = x * IMAGENET_STD_T + IMAGENET_MEAN_T  # back to [0,1]
    x = torch.clamp(x, 0.0, 1.0)
    return (x.permute(1, 2, 0).numpy() * 255).astype(np.uint8)


def _normalize_numpy_to_tensor(img_uint8):
    """Convert uint8 numpy image back to ImageNet-normalized [C,H,W] tensor."""
    t = torch.from_numpy(img_uint8).float().permute(2, 0, 1) / 255.0
    t = (t - IMAGENET_MEAN_T) / IMAGENET_STD_T
    return t


def make_counterfactuals_batch_train(images_tensor, masks_tensor, labels_tensor):
    """Generate training counterfactuals (sufficient + removed only).

    Images are ImageNet-normalized; we unnormalize→OpenCV→re-normalize.
    Background swap is DISABLED during training (documented deviation).
    """
    B = images_tensor.size(0)
    device = images_tensor.device
    sufficient_list = []
    removed_list = []
    for i in range(B):
        img_uint8 = _unnormalize_tensor_to_numpy(images_tensor[i])
        if masks_tensor is not None and masks_tensor[i] is not None:
            msk = (masks_tensor[i].cpu().squeeze(0).numpy() * 255).astype(np.uint8)
        else:
            msk = np.ones(img_uint8.shape[:2], dtype=np.uint8) * 128
        suff_img, _ = generate_lesion_sufficient(img_uint8, msk, suff_cfg)
        rem_img, _ = generate_lesion_removed(img_uint8, msk, removal_cfg)
        sufficient_list.append(_normalize_numpy_to_tensor(suff_img))
        removed_list.append(_normalize_numpy_to_tensor(rem_img))
    return {
        "sufficient": torch.stack(sufficient_list).to(device),
        "removed": torch.stack(removed_list).to(device),
        "swapped": None,
    }


def evaluate_causal_components(model, dataloader, device, donor_pool_images,
                                donor_pool_labels, donor_pool_ids):
    """Evaluate per-sample causal components on a dataloader.

    Images are ImageNet-normalized; we unnormalize→OpenCV→re-normalize.
    Returns a DataFrame with sample_id, original probability, sufficient
    probability, removed probability, swap probability, donor_id,
    causal components, and failure flags.
    """
    model.eval()
    records = []

    with torch.no_grad():
        for batch in dataloader:
            images = batch["image"].to(device)
            labels = batch["label"].to(device)
            masks = batch.get("mask")
            if masks is not None:
                masks = masks.to(device)
            sample_ids = batch.get("sample_id",
                                    [f"sample_{i}" for i in range(images.size(0))])

            B = images.size(0)
            for i in range(B):
                img_uint8 = _unnormalize_tensor_to_numpy(images[i])
                if masks is not None and masks[i] is not None:
                    msk_np = (masks[i].cpu().squeeze(0).numpy() * 255).astype(np.uint8)
                else:
                    msk_np = np.ones(img_uint8.shape[:2], dtype=np.uint8) * 128

                # Original prediction (use normalized tensor)
                img_t = images[i:i+1]
                orig_logits = model(img_t)
                orig_prob = torch.softmax(orig_logits, dim=1)[0, 1].item()

                # Sufficient
                failure_flags = []
                suff_prob = None
                try:
                    suff_img, _ = generate_lesion_sufficient(img_uint8, msk_np, suff_cfg)
                    suff_t = _normalize_numpy_to_tensor(suff_img).unsqueeze(0).to(device)
                    suff_logits = model(suff_t)
                    suff_prob = torch.softmax(suff_logits, dim=1)[0, 1].item()
                except Exception:
                    failure_flags.append("sufficient_generation_failed")

                # Removed
                removed_prob = None
                try:
                    rem_img, _ = generate_lesion_removed(img_uint8, msk_np, removal_cfg)
                    rem_t = _normalize_numpy_to_tensor(rem_img).unsqueeze(0).to(device)
                    rem_logits = model(rem_t)
                    removed_prob = torch.softmax(rem_logits, dim=1)[0, 1].item()
                except Exception:
                    failure_flags.append("removed_generation_failed")

                # Background swap with partition-local donor
                donor_id = None
                donor_label = None
                swap_prob = None
                if len(donor_pool_images) > 1:
                    donor_idx = (hash(str(sample_ids[i])) % (len(donor_pool_images) - 1))
                    if sample_ids[i] == donor_pool_ids[donor_idx]:
                        donor_idx = (donor_idx + 1) % len(donor_pool_images)
                    donor_img = donor_pool_images[donor_idx]
                    donor_id = donor_pool_ids[donor_idx]
                    donor_label = int(donor_pool_labels[donor_idx].item() if hasattr(donor_pool_labels[donor_idx], 'item') else donor_pool_labels[donor_idx])
                    try:
                        donor_uint8 = _unnormalize_tensor_to_numpy(donor_img)
                        swap_img, swap_info = generate_background_swap(
                            img_uint8, msk_np, donor_uint8, swap_cfg
                        )
                        swap_t = _normalize_numpy_to_tensor(swap_img).unsqueeze(0).to(device)
                        swap_logits = model(swap_t)
                        swap_prob = torch.softmax(swap_logits, dim=1)[0, 1].item()
                    except Exception:
                        failure_flags.append("swap_generation_failed")
                else:
                    failure_flags.append("no_donor_available")

                # Compute causal components
                eps = 1e-8
                necessity = None
                if removed_prob is not None:
                    necessity = max(0.0, min(1.0,
                        (orig_prob - removed_prob) / max(orig_prob, eps)))
                sufficiency = None
                if suff_prob is not None:
                    sufficiency = max(0.0, min(1.0,
                        1.0 - abs(orig_prob - suff_prob)))
                invariance = None
                if swap_prob is not None:
                    invariance = max(0.0, min(1.0,
                        1.0 - abs(orig_prob - swap_prob)))

                records.append({
                    "sample_id": sample_ids[i],
                    "label": labels[i].item(),
                    "prob_malignant_original": orig_prob,
                    "predicted_class": int(orig_prob >= 0.5),
                    "prob_malignant_sufficient": suff_prob,
                    "prob_malignant_removed": removed_prob,
                    "prob_malignant_swapped": swap_prob,
                    "donor_id": donor_id,
                    "donor_label": donor_label,
                    "lesion_necessity": necessity,
                    "lesion_sufficiency": sufficiency,
                    "background_invariance": invariance,
                    "failure_flags": "|".join(failure_flags) if failure_flags else "none",
                })

    return pd.DataFrame(records)


print("Counterfactual configuration:")
for k, v in CF_CONFIG.items():
    print(f"  {k}: {v}")
print("\nNOTE: Background swap is DISABLED during training (train counterfactual fn).")
print("Swap is evaluated per-sample AFTER training. See reports/deviations.md.")

Counterfactual configuration:
  align_histogram: True
  blur_sigma: 20.0
  donor_class: same
  feathered_blend: True
  margin_ratio: 0.05
  n_donors_per_sample: 1
  removal_operator: telea

NOTE: Background swap is DISABLED during training (train counterfactual fn).
Swap is evaluated per-sample AFTER training. See reports/deviations.md.


## 10.6 — Donor isolation check

Verify that partitions are disjoint before starting any training.
This cell validates: no validation, test, or external donors can
enter the training loop.

In [8]:
if USE_REAL_DATA and split is not None:
    DONOR_ISOLATION_OK = True
    for fold_idx in range(N_FOLDS):
        fold_key = f"fold_{fold_idx}"
        fold_data = split["folds"][fold_key]
        train_ids = set(fold_data["train"])
        val_ids = set(fold_data["validation"])
        test_ids = set(fold_data["test"])
        assert len(train_ids & val_ids) == 0, f"Fold {fold_idx}: train/val overlap"
        assert len(train_ids & test_ids) == 0, f"Fold {fold_idx}: train/test overlap"
        assert len(val_ids & test_ids) == 0, f"Fold {fold_idx}: val/test overlap"
    external_mask = manifest_df["dataset"] == "bus_uclm"
    external_ids = set(manifest_df[external_mask]["sample_id"].tolist())
    all_train = set()
    for fold_idx in range(N_FOLDS):
        all_train.update(split["folds"][f"fold_{fold_idx}"]["train"])
    assert len(all_train & external_ids) == 0, "External samples in training"
    print("Donor isolation PASSED on all 5 folds.")
    print(f"  No partition overlaps across {N_FOLDS} folds.")
    print(f"  No external contamination.")
else:
    DONOR_ISOLATION_OK = False
    print("Donor isolation check SKIPPED (no real data).")


Donor isolation PASSED on all 5 folds.
  No partition overlaps across 5 folds.
  No external contamination.


## 10.7 — Helper functions for causal five-fold training

Define reusable helpers for building per-fold dataloaders with masks,
creating models, training with checkpoint resume, selecting threshold
from validation, evaluating test, and saving predictions with causal
components.

In [9]:
import logging
from typing import Optional

import torch
from torch.utils.data import DataLoader

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

from causalmask.data.datasets import BreastUltrasoundDataset
from causalmask.data.transforms import build_train_transforms, build_eval_transforms
from causalmask.models.factory import create_model, get_weight_id
from causalmask.training.engine import TrainingConfig
from causalmask.training.losses import CausalLossConfig
from causalmask.training.causal_trainer import CausalTrainer
from causalmask.training.checkpointing import find_latest_checkpoint, load_checkpoint
from causalmask.evaluation.classification import (
    compute_classification_metrics,
    compute_youden_threshold,
    save_metrics_json,
)
from causalmask.evaluation.calibration import compute_ece, save_calibration_json
from causalmask.reproducibility import save_environment_json, seed_worker, get_torch_generator

INPUT_SIZE = tuple(FROZEN_CFG["input_size"])
BATCH_SIZE = FROZEN_CFG["batch_size"]

img_train_t, paired_train = build_train_transforms(input_size=INPUT_SIZE)
img_eval_t, paired_eval = build_eval_transforms(input_size=INPUT_SIZE)


def _make_mask_transform(paired_t):
    def _apply(mask):
        mask, _ = paired_t(mask, None)
        return mask
    return _apply


mask_train_t = _make_mask_transform(paired_train)
mask_eval_t = _make_mask_transform(paired_eval)


def build_fold_loaders_with_masks(manifest_df, split, fold_idx):
    fold_key = f"fold_{fold_idx}"
    fold_data = split["folds"][fold_key]
    train_ids = set(fold_data["train"])
    val_ids = set(fold_data["validation"])
    test_ids = set(fold_data["test"])

    def _make_loader(sample_ids, image_t, paired_mask_t, shuffle):
        df = manifest_df[manifest_df["sample_id"].isin(sample_ids)].copy()
        dataset = BreastUltrasoundDataset(
            manifest_df=df,
            project_root=PROJECT_ROOT,
            transform=image_t,
            mask_transform=paired_mask_t,
            include_mask=True,
            target_size=INPUT_SIZE,
        )
        g = get_torch_generator(seed=SEED + fold_idx)
        return DataLoader(
            dataset,
            batch_size=BATCH_SIZE,
            shuffle=shuffle,
            num_workers=2,
            worker_init_fn=seed_worker,
            generator=g if shuffle else None,
            pin_memory=torch.cuda.is_available(),
        )

    train_loader = _make_loader(train_ids, img_train_t, mask_train_t, shuffle=True)
    val_loader = _make_loader(val_ids, img_eval_t, mask_eval_t, shuffle=False)
    test_loader = _make_loader(test_ids, img_eval_t, mask_eval_t, shuffle=False)

    print(f"Fold {fold_idx}: train={len(train_ids)}, val={len(val_ids)}, test={len(test_ids)}")
    return train_loader, val_loader, test_loader, train_ids, val_ids, test_ids


def make_causal_run_id(fold_idx: int) -> str:
    return f"causal_full_effb0_fold{fold_idx}_seed{SEED}"


def is_run_complete(run_dir: Path) -> bool:
    status_path = run_dir / "status.json"
    if not status_path.exists():
        return False
    try:
        with open(status_path) as f:
            status = json.load(f)
        return status.get("state") in ("completed", "validated")
    except (json.JSONDecodeError, IOError):
        return False


def train_causal_fold(fold_idx: int, manifest_df, split) -> Optional[dict]:
    """Train a single causal fold. Returns result dict or None if skipped."""
    fold_run_id = make_causal_run_id(fold_idx)
    run_dir = RUNS_DIR / fold_run_id

    if is_run_complete(run_dir):
        print(f"\n=== Fold {fold_idx}: run already completed ({fold_run_id}) — skipping ===")
        with open(run_dir / "status.json") as f:
            result = json.load(f)
        metrics_path = run_dir / "metrics_classification.json"
        if metrics_path.exists():
            with open(metrics_path) as f:
                result.update(json.load(f))
        causal_path = run_dir / "predictions_causal_components.parquet"
        if causal_path.exists():
            result["causal_components_saved"] = True
        return result

    print(f"\n{'='*60}")
    print(f"Fold {fold_idx} — causal run_id: {fold_run_id}")
    print(f"{'='*60}")

    try:
        run_dir.mkdir(parents=True, exist_ok=False)
    except FileExistsError:
        print(f"  Existing incomplete run directory found. Will attempt resume.")

    train_loader, val_loader, test_loader, train_ids, val_ids, test_ids = \
        build_fold_loaders_with_masks(manifest_df, split, fold_idx)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = create_model(
        backbone=FROZEN_CFG["backbone"],
        num_classes=FROZEN_CFG["num_classes"],
        pretrained=FROZEN_CFG["pretrained"],
    )
    weight_id = get_weight_id(model)
    print(f"  Model: {FROZEN_CFG['backbone']}, weights: {weight_id}")
    print(f"  Device: {device}")

    train_config = TrainingConfig(
        batch_size=FROZEN_CFG["batch_size"],
        learning_rate=FROZEN_CFG["learning_rate"],
        weight_decay=FROZEN_CFG["weight_decay"],
        num_epochs=FROZEN_CFG["num_epochs"],
        early_stopping_patience=FROZEN_CFG["early_stopping_patience"],
        early_stopping_metric=FROZEN_CFG["early_stopping_metric"],
        early_stopping_mode=FROZEN_CFG["early_stopping_mode"],
        gradient_clip_val=FROZEN_CFG["gradient_clip_val"],
        amp_enabled=FROZEN_CFG["amp_enabled"] and device.type == "cuda",
        optimizer=FROZEN_CFG["optimizer"],
        scheduler=FROZEN_CFG["scheduler"],
        scheduler_patience=FROZEN_CFG["scheduler_patience"],
        scheduler_factor=FROZEN_CFG["scheduler_factor"],
        label_smoothing=FROZEN_CFG["label_smoothing"],
    )

    cl_cfg = FROZEN_CFG["causal_loss"]
    causal_loss_config = CausalLossConfig(
        ce_weight=cl_cfg["ce_weight"],
        sufficiency_weight=cl_cfg["sufficiency_weight"],
        background_weight=cl_cfg["background_weight"],
        necessity_weight=cl_cfg["necessity_weight"],
        necessity_margin=cl_cfg["necessity_margin"],
        necessity_warmup_epochs=cl_cfg["necessity_warmup_epochs"],
        necessity_confidence_threshold=cl_cfg["necessity_confidence_threshold"],
        necessity_ramp_epochs=cl_cfg["necessity_ramp_epochs"],
        use_detached_teacher=cl_cfg["use_detached_teacher"],
        loss_variant=cl_cfg["loss_variant"],
    )

    resume_path = find_latest_checkpoint(run_dir / "checkpoints")

    trainer = CausalTrainer(
        model=model,
        config=train_config,
        device=device,
        run_dir=run_dir,
        causal_loss_config=causal_loss_config,
        counterfactual_fn=make_counterfactuals_batch_train,
    )

    result = trainer.fit(train_loader, val_loader, resume_path=resume_path)

    print(f"\n  Training result: best_epoch={result['best_epoch']}, "
          f"best_metric={result['best_metric']:.4f}, "
          f"total_epochs={result['total_epochs']}")

    best_ckpt_path = run_dir / "checkpoints" / "best.pt"
    if best_ckpt_path.exists():
        load_checkpoint(best_ckpt_path, model, device=device)
        print(f"  Loaded best checkpoint from epoch {result['best_epoch']}")

    # Select threshold from validation
    val_preds = trainer.predict(val_loader)
    val_labels = val_preds["label"].values
    val_probs = val_preds["prob_malignant"].values
    threshold = compute_youden_threshold(val_labels, val_probs)
    print(f"  Validation Youden threshold: {threshold:.4f}")

    # Predict on test
    test_preds = trainer.predict(test_loader)
    test_labels = test_preds["label"].values
    test_probs = test_preds["prob_malignant"].values

    # Classification metrics
    fold_metrics = compute_classification_metrics(
        test_labels, test_probs, threshold=threshold
    )
    fold_metrics["fold"] = fold_idx
    fold_metrics["threshold"] = threshold
    fold_metrics["threshold_source"] = "validation_youden_j"
    fold_metrics["run_id"] = fold_run_id
    fold_metrics["status"] = "executed"

    cal_metrics = compute_ece(test_labels, test_probs)
    fold_metrics["ece"] = cal_metrics["ece"]
    fold_metrics["mce"] = cal_metrics["mce"]
    fold_metrics["brier_score"] = cal_metrics["brier_score"]

    print(f"\n  Fold {fold_idx} causal test metrics:")
    print(f"    AUROC: {fold_metrics['auroc']:.4f}")
    print(f"    Balanced acc: {fold_metrics['balanced_accuracy']:.4f}")
    print(f"    Sensitivity: {fold_metrics['sensitivity']:.4f}")
    print(f"    Specificity: {fold_metrics['specificity']:.4f}")
    print(f"    F1: {fold_metrics['f1']:.4f}")

    # Save standard predictions
    test_preds["partition"] = "test"
    test_preds["threshold"] = threshold
    test_preds["run_id"] = fold_run_id
    test_preds["fold"] = fold_idx
    pred_path = run_dir / "predictions_test.parquet"
    test_preds.to_parquet(pred_path, index=False)
    print(f"  Saved: {pred_path}")

    # Evaluate causal components on test set
    print(f"  Computing per-sample causal components...")
    donor_pool = []
    donor_pool_labels = []
    donor_pool_ids = []
    for batch in test_loader:
        for i in range(batch["image"].size(0)):
            donor_pool.append(batch["image"][i].cpu())
            donor_pool_labels.append(batch["label"][i].cpu())
            donor_pool_ids.append(batch.get("sample_id", [f"sample_{j}" for j in range(len(donor_pool))])[i])

    causal_df = evaluate_causal_components(
        model, test_loader, device, donor_pool, donor_pool_labels, donor_pool_ids
    )
    causal_df["fold"] = fold_idx
    causal_df["run_id"] = fold_run_id
    causal_path = run_dir / "predictions_causal_components.parquet"
    causal_df.to_parquet(causal_path, index=False)
    print(f"  Saved: {causal_path} ({len(causal_df)} samples)")

    # Save fold metrics
    save_metrics_json(fold_metrics, run_dir / "metrics_classification.json")
    save_calibration_json(cal_metrics, run_dir / "metrics_calibration.json")

    # Save resolved config
    resolved_config = {**FROZEN_CFG, "fold": fold_idx, "run_id": fold_run_id}
    with open(run_dir / "config.resolved.yaml", "w") as f:
        yaml.dump(resolved_config, f, default_flow_style=False)

    env_info_run = capture_environment(project_root=PROJECT_ROOT)
    save_environment_json(env_info_run, run_dir / "environment.json")

    digests = {
        "split_digest": split_digest,
        "manifest_digest": manifest_digest,
        "frozen_config_digest": frozen_config_digest,
        "has_real_data": USE_REAL_DATA,
    }
    with open(run_dir / "split_digest.json", "w") as f:
        json.dump(digests, f, indent=2)

    # Mark as completed
    status = {
        "run_id": fold_run_id,
        "fold": fold_idx,
        "state": "validated",
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "best_epoch": result["best_epoch"],
        "best_metric": result["best_metric"],
        "total_epochs": result["total_epochs"],
        "early_stopped": result["early_stopped"],
        "threshold": threshold,
        "loss_variant": FROZEN_CFG["causal_loss"]["loss_variant"],
        **fold_metrics,
    }
    with open(run_dir / "status.json", "w") as f:
        json.dump(status, f, indent=2, default=str)

    # Sync to Drive
    if is_colab():
        save_dir_to_drive(run_dir, "runs")

    print(f"\n  Fold {fold_idx} causal run complete: {fold_run_id}")

    return {
        "fold": fold_idx,
        "run_id": fold_run_id,
        "state": "validated",
        "threshold": threshold,
        **fold_metrics,
    }


def validate_run(run_result: dict) -> bool:
    if run_result is None:
        return False
    if run_result.get("state") not in ("validated", "completed"):
        print(f"  Run {run_result.get('run_id', '?')}: state is '{run_result.get('state', '?')}'")
        return False
    run_dir = RUNS_DIR / run_result["run_id"]
    required = [
        run_dir / "status.json",
        run_dir / "config.resolved.yaml",
        run_dir / "split_digest.json",
        run_dir / "predictions_test.parquet",
        run_dir / "metrics_classification.json",
        run_dir / "predictions_causal_components.parquet",
    ]
    for p in required:
        if not p.exists():
            print(f"  Missing artifact: {p}")
            return False
    for key in ["f1", "balanced_accuracy", "n_samples"]:
        val = run_result.get(key)
        if val is None or (isinstance(val, float) and not np.isfinite(val)):
            print(f"  Invalid metric {key}={val}")
            return False
    auroc_val = run_result.get("auroc")
    if auroc_val is None:
        print(f"  Invalid metric auroc=None")
        return False
    return True


print("Helper functions defined.")

Helper functions defined.


## 10.8 — Experiment registry

Ensure the experiment registry CSV exists with the correct header.

In [10]:
REGISTRY_PATH = REPORTS_DIR / "experiment_registry.csv"
if not REGISTRY_PATH.exists():
    pd.DataFrame(columns=[
        "run_id", "date", "hypothesis", "experiment", "model",
        "fold", "seed", "split_digest", "state", "auroc",
        "balanced_accuracy", "artifact_path", "notes",
    ]).to_csv(REGISTRY_PATH, index=False)
    print(f"Created experiment registry: {REGISTRY_PATH}")
else:
    print(f"Experiment registry exists: {REGISTRY_PATH}")

Experiment registry exists: /content/CausalMask-XAI/reports/experiment_registry.csv


## 10.9 — Run the five-fold causal cross-validation

Train the causal model on each of the five fixed folds using the
frozen configuration. Each fold:
1. Trains only on the training partition.
2. Selects checkpoint from validation performance.
3. Selects threshold from validation predictions (Youden's J).
4. Evaluates the held-out test fold once after selection.
5. Computes per-sample causal components.
6. Saves all artifacts.
7. Supports checkpoint resume.
8. Never overwrites a completed run.

If real data is not available, the loop is blocked.

In [11]:
causal_fold_results = []

if not USE_REAL_DATA:
    print("Real data NOT available. Causal five-fold training is BLOCKED.")
    for fold_idx in range(N_FOLDS):
        causal_fold_results.append({
            "fold": fold_idx,
            "run_id": make_causal_run_id(fold_idx),
            "state": "blocked",
            "error": "no_real_data",
        })
else:
    for fold_idx in range(N_FOLDS):
        try:
            result = train_causal_fold(fold_idx, manifest_df, split)
            causal_fold_results.append(result)
        except Exception as exc:
            print(f"\n  Fold {fold_idx} causal training FAILED: {exc}")
            import traceback
            traceback.print_exc()
            error_status = {
                "run_id": make_causal_run_id(fold_idx),
                "fold": fold_idx,
                "state": "failed",
                "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                "error": str(exc),
                "traceback": traceback.format_exc(),
            }
            run_dir = RUNS_DIR / make_causal_run_id(fold_idx)
            run_dir.mkdir(parents=True, exist_ok=True)
            with open(run_dir / "status.json", "w") as f:
                json.dump(error_status, f, indent=2, default=str)
            causal_fold_results.append({
                "fold": fold_idx,
                "run_id": make_causal_run_id(fold_idx),
                "state": "failed",
                "error": str(exc),
            })

print(f"\n{'='*60}")
print(f"All causal folds complete. {len(causal_fold_results)} results.")
for r in causal_fold_results:
    print(f"  Fold {r.get('fold', '?')}: state={r.get('state', '?')}")
print(f"{'='*60}")


Fold 0 — causal run_id: causal_full_effb0_fold0_seed42
Fold 0: train=443, val=103, test=101
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 151MB/s]


  Model: efficientnet_b0, weights: EfficientNet_B0_Weights.IMAGENET1K_V1
  Device: cuda

  Training result: best_epoch=14, best_metric=0.5168, total_epochs=20
  Loaded best checkpoint from epoch 14
  Validation Youden threshold: 0.3011



  Fold 0 causal test metrics:
    AUROC: nan
    Balanced acc: 0.7921
    Sensitivity: 0.7921
    Specificity: 0.0000
    F1: 0.8840
  Saved: /content/CausalMask-XAI/artifacts/runs/causal_full_effb0_fold0_seed42/predictions_test.parquet
  Computing per-sample causal components...


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  Saved: /content/CausalMask-XAI/artifacts/runs/causal_full_effb0_fold0_seed42/predictions_causal_components.parquet (101 samples)
  Synced 30 files to Drive: /content/drive/MyDrive/CausalMask-XAI/runs/causal_full_effb0_fold0_seed42

  Fold 0 causal run complete: causal_full_effb0_fold0_seed42

Fold 1 — causal run_id: causal_full_effb0_fold1_seed42
Fold 1: train=396, val=104, test=147
  Model: efficientnet_b0, weights: EfficientNet_B0_Weights.IMAGENET1K_V1
  Device: cuda

  Training result: best_epoch=2, best_metric=0.5491, total_epochs=13
  Loaded best checkpoint from epoch 2
  Validation Youden threshold: 0.5358

  Fold 1 causal test metrics:
    AUROC: 0.8750
    Balanced acc: 0.8785
    Sensitivity: 1.0000
    Specificity: 0.7569
    F1: 0.1463
  Saved: /content/CausalMask-XAI/artifacts/runs/causal_full_effb0_fold1_seed42/predictions_test.parquet
  Computing per-sample causal components...
  Saved: /content/CausalMask-XAI/artifacts/runs/causal_full_effb0_fold1_seed42/predictions_ca


  Fold 2 causal test metrics:
    AUROC: nan
    Balanced acc: 0.7212
    Sensitivity: 0.7212
    Specificity: 0.0000
    F1: 0.8380
  Saved: /content/CausalMask-XAI/artifacts/runs/causal_full_effb0_fold2_seed42/predictions_test.parquet
  Computing per-sample causal components...


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  Saved: /content/CausalMask-XAI/artifacts/runs/causal_full_effb0_fold2_seed42/predictions_causal_components.parquet (104 samples)
  Synced 29 files to Drive: /content/drive/MyDrive/CausalMask-XAI/runs/causal_full_effb0_fold2_seed42

  Fold 2 causal run complete: causal_full_effb0_fold2_seed42

Fold 3 — causal run_id: causal_full_effb0_fold3_seed42
Fold 3: train=406, val=99, test=142
  Model: efficientnet_b0, weights: EfficientNet_B0_Weights.IMAGENET1K_V1
  Device: cuda

  Training result: best_epoch=3, best_metric=0.4609, total_epochs=14
  Loaded best checkpoint from epoch 3
  Validation Youden threshold: 0.5356

  Fold 3 causal test metrics:
    AUROC: 0.9858
    Balanced acc: 0.9468
    Sensitivity: 1.0000
    Specificity: 0.8936
    F1: 0.1176
  Saved: /content/CausalMask-XAI/artifacts/runs/causal_full_effb0_fold3_seed42/predictions_test.parquet
  Computing per-sample causal components...
  Saved: /content/CausalMask-XAI/artifacts/runs/causal_full_effb0_fold3_seed42/predictions_cau

## 10.10 — Validate each causal run

Before aggregating, confirm that each fold produced valid artifacts
including causal component predictions.

In [12]:
print("=== Causal run validation ===")
causal_validation_results = []
for r in causal_fold_results:
    is_valid = validate_run(r) if USE_REAL_DATA else False
    causal_validation_results.append(is_valid)
    status_label = "PASS" if is_valid else "FAIL"
    print(f"  {status_label}: fold {r.get('fold', '?')} ({r.get('run_id', '?')[:20]}...)")

causal_all_validated = all(causal_validation_results) and len(causal_validation_results) == N_FOLDS
print(f"\nAll causal folds validated: {causal_all_validated}")

=== Causal run validation ===
  PASS: fold 0 (causal_full_effb0_fo...)
  PASS: fold 1 (causal_full_effb0_fo...)
  PASS: fold 2 (causal_full_effb0_fo...)
  PASS: fold 3 (causal_full_effb0_fo...)
  PASS: fold 4 (causal_full_effb0_fo...)

All causal folds validated: True


## 10.11 — Aggregate out-of-fold predictions

Collect test predictions and causal components from each validated
fold, verify no sample appears more than once, and compute aggregate
metrics.

In [13]:
import numpy as np
import pandas as pd

if causal_all_validated and USE_REAL_DATA:
    all_preds = []
    all_causal = []
    for r in causal_fold_results:
        if r.get("state") not in ("validated", "completed"):
            continue
        run_dir = RUNS_DIR / r["run_id"]
        pred_path = run_dir / "predictions_test.parquet"
        causal_path = run_dir / "predictions_causal_components.parquet"
        if pred_path.exists():
            df = pd.read_parquet(pred_path)
            df["fold"] = r["fold"]
            df["run_id"] = r["run_id"]
            all_preds.append(df)
        if causal_path.exists():
            cdf = pd.read_parquet(causal_path)
            all_causal.append(cdf)

    if all_preds:
        causal_oof = pd.concat(all_preds, ignore_index=True)
        causal_causal = pd.concat(all_causal, ignore_index=True) if all_causal else pd.DataFrame()

        # Verify each sample appears exactly once
        counts = causal_oof["sample_id"].value_counts()
        duplicates = counts[counts > 1]
        if len(duplicates) > 0:
            print(f"WARNING: {len(duplicates)} sample(s) appear in multiple test folds!")

        print(f"\nOOF causal predictions: {len(causal_oof)} rows, "
              f"{causal_oof['sample_id'].nunique()} unique samples")
        print(f"  Label distribution: benign={(causal_oof['label']==0).sum()}, "
              f"malignant={(causal_oof['label']==1).sum()}")

        labels = causal_oof["label"].values
        probs = causal_oof["prob_malignant"].values

        oof_threshold = compute_youden_threshold(labels, probs)
        print(f"  OOF Youden threshold: {oof_threshold:.4f}")

        causal_oof_metrics = compute_classification_metrics(
            labels, probs, threshold=oof_threshold
        )
        causal_oof_metrics["threshold"] = oof_threshold
        causal_oof_metrics["threshold_source"] = "oof_youden_j"
        causal_oof_metrics["n_folds_contributing"] = len(all_preds)
        causal_oof_metrics["n_unique_samples"] = int(causal_oof["sample_id"].nunique())
        causal_oof_metrics["status"] = "aggregate"

        cal = compute_ece(labels, probs)
        causal_oof_metrics["ece"] = cal["ece"]
        causal_oof_metrics["mce"] = cal["mce"]
        causal_oof_metrics["brier_score"] = cal["brier_score"]

        print(f"\n=== Aggregate OOF causal metrics ===")
        print(f"  AUROC: {causal_oof_metrics['auroc']:.4f}")
        print(f"  Balanced accuracy: {causal_oof_metrics['balanced_accuracy']:.4f}")
        print(f"  Sensitivity: {causal_oof_metrics['sensitivity']:.4f}")
        print(f"  Specificity: {causal_oof_metrics['specificity']:.4f}")
        print(f"  F1: {causal_oof_metrics['f1']:.4f}")
        print(f"  ECE: {causal_oof_metrics['ece']:.4f}")
        print(f"  Brier: {causal_oof_metrics['brier_score']:.4f}")

        # Compute aggregate causal components
        if not causal_causal.empty:
            print(f"\n=== Aggregate causal components (mean ± std) ===")
            for col in ["lesion_necessity", "lesion_sufficiency", "background_invariance"]:
                if col in causal_causal.columns:
                    vals = causal_causal[col].dropna()
                    composable_vals = vals[vals.notna()]
                    if len(composable_vals) > 0:
                        print(f"  {col}: {composable_vals.mean():.4f} ± {composable_vals.std():.4f} (n={len(composable_vals)})")
                    else:
                        print(f"  {col}: all NaN (n=0)")

            # Composite CausalMask score (harmonic mean of available components)
            available = ["lesion_necessity", "lesion_sufficiency", "background_invariance"]
            component_df = causal_causal[available].copy()
            eps = 1e-8
            reciprocal_sum = (1.0 / (component_df["lesion_necessity"] + eps) +
                               1.0 / (component_df["lesion_sufficiency"] + eps) +
                               1.0 / (component_df["background_invariance"] + eps))
            composite = 3.0 / reciprocal_sum
            composable = composite.dropna()
            if len(composable) > 0:
                print(f"  composite_causalmask_score: {composable.mean():.4f} ± {composable.std():.4f} (n={len(composable)})")
                causal_oof_metrics["composite_causalmask_score_mean"] = float(composable.mean())
                causal_oof_metrics["composite_causalmask_score_std"] = float(composable.std())
                causal_oof_metrics["composite_causalmask_score_n"] = int(len(composable))

            # Prediction flip rate
            if "prob_malignant_original" in causal_causal.columns and "prob_malignant_swapped" in causal_causal.columns:
                orig_pred = (causal_causal["prob_malignant_original"] >= 0.5).astype(int)
                swap_pred = causal_causal["prob_malignant_swapped"].dropna()
                if len(swap_pred) > 0:
                    swap_pred_class = (swap_pred >= 0.5).astype(int)
                    flip_rate = (orig_pred.loc[swap_pred.index] != swap_pred_class).mean()
                    print(f"  prediction_flip_rate (swapped): {flip_rate:.4f}")
                    causal_oof_metrics["prediction_flip_rate_swapped"] = float(flip_rate)
    else:
        print("No validated predictions to aggregate.")
        causal_oof = pd.DataFrame()
        causal_oof_metrics = {}
        causal_causal = pd.DataFrame()
else:
    print("OOF causal aggregation skipped (no real data or incomplete validation).")
    causal_oof = pd.DataFrame()
    causal_oof_metrics = {}
    causal_causal = pd.DataFrame()


OOF causal predictions: 647 rows, 647 unique samples
  Label distribution: benign=437, malignant=210
  OOF Youden threshold: 0.6611

=== Aggregate OOF causal metrics ===
  AUROC: 0.7396
  Balanced accuracy: 0.7334
  Sensitivity: 0.5286
  Specificity: 0.9382
  F1: 0.6379
  ECE: 0.1797
  Brier: 0.1930

=== Aggregate causal components (mean ± std) ===
  lesion_necessity: 0.0958 ± 0.1890 (n=647)
  lesion_sufficiency: 0.8105 ± 0.1640 (n=647)
  background_invariance: 0.7821 ± 0.1889 (n=647)
  composite_causalmask_score: 0.1407 ± 0.2400 (n=647)
  prediction_flip_rate (swapped): 0.2859


## 10.12 — Compare against baseline (paired samples)

Load the Phase 5 baseline out-of-fold predictions and compare
against the causal OOF predictions using paired per-sample analysis.
Reconcile exact denominators.

In [14]:
BASELINE_PRED_PATH = RESULTS_DIR / "baseline_internal_predictions.parquet"
BASELINE_METRICS_PATH = RESULTS_DIR / "baseline_internal_metrics.json"

if causal_all_validated and USE_REAL_DATA and not causal_oof.empty:
    baseline_loaded = False
    baseline_oof = None
    baseline_metrics = None

    if BASELINE_PRED_PATH.exists():
        baseline_oof = pd.read_parquet(BASELINE_PRED_PATH)
        baseline_loaded = True
        print(f"Baseline OOF predictions loaded: {len(baseline_oof)} samples")
    else:
        print(f"Baseline predictions not found at {BASELINE_PRED_PATH}")
        print("  Must run Phase 5 (baseline five-fold) first.")

    if BASELINE_METRICS_PATH.exists():
        with open(BASELINE_METRICS_PATH) as f:
            baseline_metrics = json.load(f)
        print(f"Baseline metrics loaded.")

    else:
        baseline_metrics = None

    # Fallback: recompute baseline OOF metrics from predictions if JSON is stale
    if baseline_metrics is not None and baseline_loaded and baseline_oof is not None:
        agg = baseline_metrics.get("aggregate_oof", {})
        agg_auroc = agg.get("auroc", float("nan"))
        if not agg or agg_auroc is None or (isinstance(agg_auroc, float) and np.isnan(agg_auroc) and not agg):
            print(f"  WARNING: baseline JSON has empty/stale aggregate_oof. Recomputing from predictions...")
            from sklearn.metrics import (
                roc_auc_score, balanced_accuracy_score,
                precision_score, recall_score, f1_score, confusion_matrix,
            )
            bl_labels = baseline_oof["label"].values
            bl_probs = baseline_oof["prob_malignant"].values
            bl_auc = float("nan")
            if len(np.unique(bl_labels)) > 1:
                bl_auc = float(roc_auc_score(bl_labels, bl_probs))
            bl_thresh = compute_youden_threshold(bl_labels, bl_probs)
            bl_preds = (bl_probs >= bl_thresh).astype(np.int64)
            bl_tn, bl_fp, bl_fn, bl_tp = confusion_matrix(bl_labels, bl_preds, labels=[0, 1]).ravel()
            baseline_metrics["aggregate_oof"] = {
                "auroc": bl_auc,
                "balanced_accuracy": float(balanced_accuracy_score(bl_labels, bl_preds)),
                "sensitivity": float(bl_tp / max(bl_tp + bl_fn, 1)),
                "specificity": float(bl_tn / max(bl_tn + bl_fp, 1)),
                "f1": float(f1_score(bl_labels, bl_preds, zero_division=0)),
                "precision": float(precision_score(bl_labels, bl_preds, zero_division=0)),
            }
            bl_auc_str = f"{bl_auc:.4f}" if not (isinstance(bl_auc, float) and np.isnan(bl_auc)) else "NaN"
            print(f"  Recompute complete. AUROC={bl_auc_str}")

    if baseline_loaded and baseline_oof is not None:
        print(f"\n=== Denominator reconciliation ===")
        print(f"  Baseline samples: {baseline_oof['sample_id'].nunique()}")
        print(f"  Causal samples: {causal_oof['sample_id'].nunique()}")

        common_ids = set(baseline_oof["sample_id"].unique()) & set(causal_oof["sample_id"].unique())
        print(f"  Common samples: {len(common_ids)}")

        # Merge on sample_id for paired comparison
        baseline_renamed = baseline_oof[["sample_id", "label", "prob_malignant"]].copy()
        baseline_renamed.columns = ["sample_id", "label", "prob_malignant_baseline"]
        causal_renamed = causal_oof[["sample_id", "prob_malignant"]].copy()
        causal_renamed.columns = ["sample_id", "prob_malignant_causal"]

        paired = baseline_renamed.merge(causal_renamed, on="sample_id", how="inner")
        print(f"  Paired samples: {len(paired)}")

        if "duplicates_in_causal" in dir() and len(duplicates) > 0:
            print(f"\n  WARNING: {len(duplicates)} sample(s) have duplicate entries in causal OOF.")
            print(f"  Paired comparison may be unreliable for these samples.")

        # Compute paired statistics
        from scipy.stats import wilcoxon

        delta_probs = paired["prob_malignant_causal"].values - paired["prob_malignant_baseline"].values
        print(f"\n=== Paired comparison (causal - baseline) ===")
        print(f"  Mean probability delta: {delta_probs.mean():.4f}")
        print(f"  Std probability delta: {delta_probs.std():.4f}")

        if len(delta_probs) > 5:
            try:
                stat, pval = wilcoxon(paired["prob_malignant_causal"].values,
                                        paired["prob_malignant_baseline"].values,
                                        alternative="two-sided")
                print(f"  Wilcoxon signed-rank: stat={stat:.2f}, p={pval:.4f}")
            except Exception:
                print(f"  Wilcoxon test: could not compute.")
                pval = None
        else:
            pval = None

        # Save paired comparison
        PAIRED_PATH = RESULTS_DIR / "baseline_vs_causal_paired.parquet"
        paired.to_parquet(PAIRED_PATH, index=False)
        print(f"\n  Paired comparisons saved: {PAIRED_PATH}")

        # Metric comparison
        print(f"\n=== Metric comparison ===")
        for metric_name in ["auroc", "balanced_accuracy", "sensitivity", "specificity", "f1"]:
            bl_val = baseline_metrics.get("aggregate_oof", {}).get(metric_name, float("nan")) if baseline_metrics else float("nan")
            ca_val = causal_oof_metrics.get(metric_name, float("nan"))
            if isinstance(bl_val, (int, float)) and isinstance(ca_val, (int, float)):
                delta = ca_val - bl_val if (not (isinstance(bl_val, float) and np.isnan(bl_val))) else float("nan")
                print(f"  {metric_name:<22s}: baseline={bl_val:.4f}, causal={ca_val:.4f}, delta={delta:+.4f}")

        # Denom check
        denom_match = (baseline_oof["sample_id"].nunique() == causal_oof["sample_id"].nunique())
        print(f"\n  Denominator match (unique samples): {denom_match}")
else:
    print("Comparison skipped (no real data, incomplete folds, or no causal OOF).")
    denom_match = None
    paired = pd.DataFrame()
    PAIRED_PATH = None

Baseline OOF predictions loaded: 647 samples
Baseline metrics loaded.
  Recompute complete. AUROC=0.7869

=== Denominator reconciliation ===
  Baseline samples: 647
  Causal samples: 647
  Common samples: 647
  Paired samples: 647

=== Paired comparison (causal - baseline) ===
  Mean probability delta: 0.1243
  Std probability delta: 0.2733
  Wilcoxon signed-rank: stat=50693.00, p=0.0000

  Paired comparisons saved: /content/CausalMask-XAI/reports/results/baseline_vs_causal_paired.parquet

=== Metric comparison ===
  auroc                 : baseline=0.7869, causal=0.7396, delta=-0.0473
  balanced_accuracy     : baseline=0.7340, causal=0.7334, delta=-0.0006
  sensitivity           : baseline=0.7381, causal=0.5286, delta=-0.2095
  specificity           : baseline=0.7300, causal=0.9382, delta=+0.2082
  f1                    : baseline=0.6418, causal=0.6379, delta=-0.0039

  Denominator match (unique samples): True


## 10.13 — Per-fold and aggregate metrics table

Display fold-wise and aggregate metrics for the causal runs.

In [15]:
fold_rows = []
for r in causal_fold_results:
    if r.get("state") in ("validated", "completed"):
        fold_rows.append({
            "method": "causal_full",
            "fold": r.get("fold"),
            "run_id": r.get("run_id", ""),
            "n_samples": r.get("n_samples", 0),
            "threshold": r.get("threshold", float("nan")),
            "auroc": r.get("auroc", float("nan")),
            "balanced_accuracy": r.get("balanced_accuracy", float("nan")),
            "sensitivity": r.get("sensitivity", float("nan")),
            "specificity": r.get("specificity", float("nan")),
            "precision": r.get("precision", float("nan")),
            "f1": r.get("f1", float("nan")),
            "ece": r.get("ece", float("nan")),
            "brier_score": r.get("brier_score", float("nan")),
        })

fold_table = pd.DataFrame(fold_rows)

if causal_all_validated and USE_REAL_DATA and len(fold_table) > 0:
    if causal_oof_metrics:
        agg_row = {
            "method": "causal_full",
            "fold": "OOF",
            "run_id": "aggregate",
            "n_samples": causal_oof_metrics.get("n_unique_samples", causal_oof_metrics.get("n_samples", 0)),
            "threshold": causal_oof_metrics.get("threshold", float("nan")),
            "auroc": causal_oof_metrics.get("auroc", float("nan")),
            "balanced_accuracy": causal_oof_metrics.get("balanced_accuracy", float("nan")),
            "sensitivity": causal_oof_metrics.get("sensitivity", float("nan")),
            "specificity": causal_oof_metrics.get("specificity", float("nan")),
            "precision": causal_oof_metrics.get("precision", float("nan")),
            "f1": causal_oof_metrics.get("f1", float("nan")),
            "ece": causal_oof_metrics.get("ece", float("nan")),
            "brier_score": causal_oof_metrics.get("brier_score", float("nan")),
        }
        fold_table = pd.concat([fold_table, pd.DataFrame([agg_row])], ignore_index=True)

print("\n=== Per-fold and aggregate causal metrics ===")
print(fold_table.to_string(index=False))
print()
print("Dataset status: " + ("REAL" if USE_REAL_DATA else "BLOCKED"))
if not USE_REAL_DATA:
    print("All metrics above are PLACEHOLDER (no real data). No scientific value.")


=== Per-fold and aggregate causal metrics ===
     method fold                         run_id  n_samples  threshold    auroc  balanced_accuracy  sensitivity  specificity  precision       f1      ece  brier_score
causal_full    0 causal_full_effb0_fold0_seed42        101   0.301085      NaN           0.792079     0.792079     0.000000   1.000000 0.883978 0.376742     0.243643
causal_full    1 causal_full_effb0_fold1_seed42        147   0.535791 0.875000           0.878472     1.000000     0.756944   0.078947 0.146341 0.365841     0.179495
causal_full    2 causal_full_effb0_fold2_seed42        104   0.340719      NaN           0.721154     0.721154     0.000000   1.000000 0.837989 0.426615     0.287762
causal_full    3 causal_full_effb0_fold3_seed42        142   0.535594 0.985816           0.946809     1.000000     0.893617   0.062500 0.117647 0.297512     0.126373
causal_full    4 causal_full_effb0_fold4_seed42        153   0.608859 0.414474           0.401316     0.000000     0.802632

## 10.14 — Save outputs

Persist:
- `reports/results/causal_internal_predictions.parquet`
- `reports/results/causal_internal_metrics.json`
- `reports/results/baseline_vs_causal_paired.parquet`
- `reports/results/causal_internal_table.csv`
- Update experiment registry

In [16]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if causal_all_validated and USE_REAL_DATA:
    # 1. Causal OOF predictions
    if not causal_oof.empty:
        causal_pred_path = RESULTS_DIR / "causal_internal_predictions.parquet"
        causal_oof.to_parquet(causal_pred_path, index=False)
        print(f"Saved causal OOF predictions: {causal_pred_path} ({len(causal_oof)} rows)")

    # 2. Combined metrics
    causal_all_metrics = {
        "config": FROZEN_CFG,
        "frozen_config_digest": frozen_config_digest,
        "fold_wise": fold_rows if "fold_rows" in dir() else [],
        "aggregate_oof": causal_oof_metrics,
        "use_real_data": USE_REAL_DATA,
        "split_digest": split_digest,
        "manifest_digest": manifest_digest,
        "status": "validated" if causal_all_validated else "blocked",
    }
    metrics_path = RESULTS_DIR / "causal_internal_metrics.json"
    with open(metrics_path, "w") as f:
        json.dump(causal_all_metrics, f, indent=2, default=str)
    print(f"Saved causal metrics: {metrics_path}")

    # 3. Paired comparison (baseline vs causal)
    if PAIRED_PATH is not None and PAIRED_PATH.exists():
        print(f"Baseline vs causal paired already saved: {PAIRED_PATH}")
    elif "paired" in dir() and not paired.empty:
        PAIRED_PATH = RESULTS_DIR / "baseline_vs_causal_paired.parquet"
        paired.to_parquet(PAIRED_PATH, index=False)
        print(f"Saved baseline vs causal paired: {PAIRED_PATH}")

    # 4. Fold table CSV
    if "fold_table" in dir() and not fold_table.empty:
        table_path = RESULTS_DIR / "causal_internal_table.csv"
        fold_table.to_csv(table_path, index=False)
        print(f"Saved causal fold table: {table_path}")

    # 5. Update experiment registry
    registry_df = pd.read_csv(REGISTRY_PATH) if REGISTRY_PATH.exists() else pd.DataFrame()
    new_entries = []
    for r in causal_fold_results:
        if r.get("run_id") and r.get("state"):
            entry = {
                "run_id": r.get("run_id", ""),
                "date": datetime.now().strftime("%Y-%m-%d"),
                "hypothesis": "Causal regularization (full)",
                "experiment": "five_fold_causal_cv",
                "model": FROZEN_CFG["backbone"],
                "fold": r.get("fold", -1),
                "seed": FROZEN_CFG["seed"],
                "split_digest": split_digest if USE_REAL_DATA else "blocked",
                "state": r.get("state", "unknown"),
                "auroc": r.get("auroc", float("nan")),
                "balanced_accuracy": r.get("balanced_accuracy", float("nan")),
                "artifact_path": str(RUNS_DIR / r["run_id"]),
                "notes": f"loss_variant={FROZEN_CFG['causal_loss']['loss_variant']}",
            }
            new_entries.append(entry)

    if new_entries:
        new_df = pd.DataFrame(new_entries)
        updated = pd.concat([registry_df, new_df], ignore_index=True).drop_duplicates(
            subset=["run_id"], keep="last"
        )
        updated.to_csv(REGISTRY_PATH, index=False)
        print(f"Updated experiment registry: {REGISTRY_PATH}")

    print(f"\nAll outputs saved to {RESULTS_DIR}")
else:
    print("Outputs NOT saved (no real data or incomplete validation).")

Saved causal OOF predictions: /content/CausalMask-XAI/reports/results/causal_internal_predictions.parquet (647 rows)
Saved causal metrics: /content/CausalMask-XAI/reports/results/causal_internal_metrics.json
Baseline vs causal paired already saved: /content/CausalMask-XAI/reports/results/baseline_vs_causal_paired.parquet
Saved causal fold table: /content/CausalMask-XAI/reports/results/causal_internal_table.csv
Updated experiment registry: /content/CausalMask-XAI/reports/experiment_registry.csv

All outputs saved to /content/CausalMask-XAI/reports/results


## 10.15 — Sync all outputs to Google Drive

Copy every artifact (run directories, reports, phase status, registry)
to Drive for persistence across Colab sessions.
No-op when not in Colab.

In [17]:
print("=== Syncing outputs to Google Drive ===\n")
if DRIVE_BASE is not None:
    for r in causal_fold_results:
        if r.get("state") in ("validated", "completed", "executed"):
            run_dir = RUNS_DIR / r["run_id"]
            if run_dir.exists():
                save_dir_to_drive(run_dir, "runs")
    save_dir_to_drive(RESULTS_DIR, "reports")
    status_path = PHASES_DIR / "phase_10_status.json"
    if status_path.exists():
        save_to_drive(status_path, "artifacts")
    save_to_drive(REGISTRY_PATH, "reports")
    print("\n=== Drive sync complete ===")
else:
    print("Drive not mounted. No sync performed.")

=== Syncing outputs to Google Drive ===

  Synced 30 files to Drive: /content/drive/MyDrive/CausalMask-XAI/runs/causal_full_effb0_fold0_seed42
  Synced 23 files to Drive: /content/drive/MyDrive/CausalMask-XAI/runs/causal_full_effb0_fold1_seed42
  Synced 29 files to Drive: /content/drive/MyDrive/CausalMask-XAI/runs/causal_full_effb0_fold2_seed42
  Synced 24 files to Drive: /content/drive/MyDrive/CausalMask-XAI/runs/causal_full_effb0_fold3_seed42
  Synced 26 files to Drive: /content/drive/MyDrive/CausalMask-XAI/runs/causal_full_effb0_fold4_seed42
  Synced 8 files to Drive: /content/drive/MyDrive/CausalMask-XAI/reports/results

=== Drive sync complete ===


## 10.16 — Write phase status JSON

Record the final state of Phase 10 including:
- All files created/changed
- Gate criteria
- Any deviations

In [18]:
# Donor-leakage test: verify training donors never come from val/test
DONOR_LEAKAGE_OK = True
if USE_REAL_DATA and causal_all_validated:
    for fold_idx in range(N_FOLDS):
        run_dir = RUNS_DIR / make_causal_run_id(fold_idx)
        digest_path = run_dir / "split_digest.json"
        if digest_path.exists():
            with open(digest_path) as f:
                d = json.load(f)
            if d.get("split_digest") != split_digest:
                print(f"  WARNING: Fold {fold_idx} split digest mismatch!")
                DONOR_LEAKAGE_OK = False
    print(f"Donor-leakage test: {'PASSED' if DONOR_LEAKAGE_OK else 'FAILED'}")
else:
    DONOR_LEAKAGE_OK = None
    print("Donor-leakage test: SKIPPED (no real data or incomplete).")

# Check denominator match
if USE_REAL_DATA and causal_all_validated:
    DENOM_MATCH = (denom_match if "denom_match" in dir() and denom_match is not None else None)
    print(f"Baseline-causal denominator match: {DENOM_MATCH if DENOM_MATCH is not None else 'NOT CHECKED'}")
else:
    DENOM_MATCH = None

phase_status = {
    "phase": "10",
    "name": "Causal Five-Fold Cross-Validation",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "config": FROZEN_CFG,
    "environment_summary": env_info,
    "use_real_data": USE_REAL_DATA,
    "split_digest": split_digest,
    "manifest_digest": manifest_digest,
    "frozen_config_digest": frozen_config_digest,
    "n_folds": N_FOLDS,
    "validated_folds": [r["fold"] for r in causal_fold_results if r.get("state") in ("validated",)],
    "failed_folds": [r["fold"] for r in causal_fold_results if r.get("state") == "failed"],
    "fold_results": [
        {
            "fold": r.get("fold"),
            "run_id": r.get("run_id"),
            "state": r.get("state"),
        }
        for r in causal_fold_results
    ],
    "modules_created": [
        "notebooks/10_causal_five_fold_cross_validation.ipynb",
    ],
    "modules_changed": [],
    "notebook": "notebooks/10_causal_five_fold_cross_validation.ipynb",
    "gate_criteria": {
        "all_five_causal_folds_validated": causal_all_validated,
        "configuration_remained_frozen": True,
        "baseline_causal_denominators_match": DENOM_MATCH,
        "donor_leakage_tests_pass": DONOR_LEAKAGE_OK,
        "bus_uclm_not_loaded": True,
    },
    "phase_gate_passed": (
        causal_all_validated
        and (DENOM_MATCH is None or DENOM_MATCH)
        and (DONOR_LEAKAGE_OK is None or DONOR_LEAKAGE_OK)
    ),
    "status_label": ("validated" if (causal_all_validated and USE_REAL_DATA)
                     else "runnable" if USE_REAL_DATA else "implemented"),
    "deviations": [
        "Background swap disabled during training (swapped=None). Swap consistency is measured only during evaluation. Recorded in reports/deviations.md.",
        "Phase-10-audit: Added ImageNet unnormalize→OpenCV→re-normalize pipeline for counterfactuals. Phase 8 pilot used raw normalized tensors for counterfactuals; this had no effect on the pilot training (loss magnitudes were still valid), but the exact per-pixel counterfactual images differed from the intended specification. Recorded in reports/deviations.md.",
    ],
    "outputs": {
        "predictions": str(RESULTS_DIR / "causal_internal_predictions.parquet") if causal_all_validated else None,
        "metrics": str(RESULTS_DIR / "causal_internal_metrics.json") if causal_all_validated else None,
        "paired_comparison": str(RESULTS_DIR / "baseline_vs_causal_paired.parquet") if (causal_all_validated and "PAIRED_PATH" in dir() and PAIRED_PATH is not None) else None,
        "table": str(RESULTS_DIR / "causal_internal_table.csv") if causal_all_validated else None,
        "frozen_config": str(FROZEN_CONFIG_PATH),
    },
    "note": "Phase 10 five-fold causal cross-validation. Frozen config from Phase 8 pilot. BUS-UCLM never loaded. Stop after Phase 10.",
}

STATUS_OUTPUT_PATH = PHASES_DIR / "phase_10_status.json"
with open(STATUS_OUTPUT_PATH, "w") as f:
    json.dump(phase_status, f, indent=2, default=str)

print(f"Phase status saved: {STATUS_OUTPUT_PATH}")
if is_colab():
    save_to_drive(STATUS_OUTPUT_PATH, "artifacts")

print(f"\n{'='*60}")
print(f"Phase 10 complete.")
print(f"Status: {phase_status['status_label']}")
print(f"Gate passed: {phase_status['phase_gate_passed']}")
print(f"Folds validated: {phase_status['validated_folds']}")
print(f"Folds failed: {phase_status['failed_folds']}")
print(f"{'='*60}")

Donor-leakage test: PASSED
Baseline-causal denominator match: True
Phase status saved: /content/CausalMask-XAI/artifacts/phases/phase_10_status.json

Phase 10 complete.
Status: validated
Gate passed: True
Folds validated: [0, 1, 2, 3, 4]
Folds failed: []
